# NB 03 · Retrieval and scoring

**Series:** Learning LangGraph through literate notebooks, notebook 3 of 7.
**Central question:** how does the assistant find the right documents, and how does it rate the evidence?

NB 02 built the skeleton of the research assistant with two stand-ins: the collectors added
fixed documents, and the score was a plain sum of authority weights. This notebook replaces both.
The collectors now search all 21 corpus documents, and the scorer applies the evidence rules agreed
in [ADR 0007](../docs/adr/0007-evidence-rules-and-search-loop.md). The structure, the state and the
contract stay as they were; the state gains one field, `claims`.

## Learning goals

After this notebook you can:

- explain how a keyword search ranks documents, and why rare words should count more than common ones;
- explain why a relevance threshold has to be measured on the corpus rather than guessed;
- let a language model extract one fact from a document as structured data, and say why the format matters;
- apply the evidence rules: authority floor, independent sources, outdated sources, contradictions and the quality gate;
- read an evidence assessment and explain why each source counted or did not.

## Setup

From this notebook on, a model reads each document and reports the figure it states. As in NB 01,
`LLM_MODE=fake` replays replies that were recorded from `qwen3:8b`, and `disabled` falls back to
them because this notebook needs a model.

In [ ]:
import datetime as dt
import math
import re
import sys
from importlib.metadata import version

from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, START, StateGraph

from research_assistant.evidence import Assessment, format_figure, parse_figure, read_claim
from research_assistant.models import get_chat_model, get_llm_mode
from research_assistant.state import ANY_NODE, ResearchState, SourceDoc, check_contract
from research_assistant.tools import load_corpus, to_source_doc

load_dotenv()

assert sys.version_info >= (3, 12), "This series needs Python 3.12 or newer."
print(f"Python {sys.version.split()[0]}")
for package in ("langgraph", "langchain-core"):
    print(f"{package} {version(package)}")

mode = get_llm_mode()
if mode == "disabled":
    mode = "fake"  # this notebook needs a model; the recorded replies stand in
llm = get_chat_model(mode, fixture="nb03_claims.yaml")
print(f"Model mode: {mode}")

## Concept: finding, reading, judging

Answering a research question takes three different kinds of work, and this notebook keeps them
apart on purpose:

| Step | Who does it | What it produces |
|---|---|---|
| **Finding** documents | plain Python: a keyword search | the documents worth reading |
| **Reading** each document | the language model | the figure the document states (a *claim*) |
| **Judging** the evidence | plain Python: fixed rules | which claims count, and whether the evidence is enough |

The model only reads. Judging stays in rules that anyone can check and that behave the same on
every run; NB 05 shows what happens when a model is asked to judge instead.

The graph keeps the shape from NB 02:

```text
          +--> [collect_primary] ----+
START ----+                          +--> [score] --> [write_answer] --> END
          +--> [collect_secondary] --+
```

The two collectors now split the work the way a researcher would. `collect_primary` looks for
**primary sources**: what a company, an authority or an association publishes about itself or from
its own data. `collect_secondary` looks for **secondary sources**: press, research, commentary,
forums.

A note on terms: for this small corpus, "primary" means original institutional or first-party
material, including an association's market survey and a foundation's own cost study. The split is
a retrieval partition for this corpus, not a general taxonomy of where sources come from; research
methods textbooks would call some of these documents independent research.

## Experiment

### 1 The corpus

Twenty-one fictional documents in six topics. Five topics hold one finance question each; the sixth
holds village club newsletters that have nothing to do with finance.

In [ ]:
CORPUS = [to_source_doc(doc) for doc in load_corpus()]

by_topic: dict[str, list[str]] = {}
for doc in CORPUS:
    by_topic.setdefault(doc["topic"], []).append(doc["source_id"])
for topic, ids in by_topic.items():
    print(f"{topic:<28} {', '.join(ids)}")

QUESTIONS = {
    "a": "What revenue did Solvane Dynamics report for the third quarter of 2025?",
    "b": "What is the implied transaction value of the merger between Talvero Bank and Nordquist Privatbank?",
    "c": "What share of client crypto assets must be kept in cold storage under the new AMSA custody rules?",
    "d": "What is Brunnhof Versicherungen's current dividend payout target?",
    "e": "What is the average annual fee of robo-advisory platforms in Switzerland in 2025?",
}

### 2 Searching by words, with rare words counting more

The search compares words. Both question and document are cut into lower-case words; short words
and words such as "the" or "what" are dropped, and a plural "s" is removed so that "platforms"
matches "platform".

Not every shared word means the same. "Solvane" appears in three documents, "2025" in twelve. A
document that shares "Solvane" with the question says far more than one that shares "2025". The
standard measure for this is the **inverse document frequency**: the fewer documents contain a
word, the higher its weight. A document's relevance is the sum of the weights of the question words
it contains.

In [ ]:
STOPWORDS = frozenset(
    "a an and are as at be by did do does for from how in is it its must of on "
    "or that the their this to under was were what which who with".split()
)


def singular(word: str) -> str:
    if len(word) > 4 and word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word


def tokenize_plain(text: str) -> set[str]:
    words = re.findall(r"[a-z0-9]+", text.lower())
    return {singular(word) for word in words if len(word) >= 3 and word not in STOPWORDS}


def word_weights(corpus: list[SourceDoc], tokenize) -> dict[str, float]:
    counts: dict[str, int] = {}
    for doc in corpus:
        for word in tokenize(f"{doc['title']}\n{doc['text']}"):
            counts[word] = counts.get(word, 0) + 1
    total = len(corpus)
    return {word: math.log((total + 1) / (count + 1)) + 1 for word, count in counts.items()}


plain_weights = word_weights(CORPUS, tokenize_plain)
for word in ("solvane", "third", "quarter", "revenue", "2025"):
    print(f"{word:<8} weight {plain_weights[word]:.2f}")

### 3 A first threshold, and why it fails

A search returns more than the documents worth reading, so it needs a threshold. The obvious one:
keep every document that reaches half the relevance of the best match. Here is the ranking for the
revenue question in topic a, with each document's relevance as a share of the best.

In [ ]:
def relevance(question: str, doc: SourceDoc, weights: dict[str, float], tokenize) -> float:
    shared = tokenize(question) & tokenize(f"{doc['title']}\n{doc['text']}")
    return sum(weights.get(word, 0.0) for word in shared)


def ranking(question: str, weights: dict[str, float], tokenize, keep=lambda doc: True):
    scores = {doc["source_id"]: relevance(question, doc, weights, tokenize) for doc in CORPUS if keep(doc)}
    best = max(scores.values())
    return sorted(((sid, round(score / best, 2)) for sid, score in scores.items() if score > 0), key=lambda pair: -pair[1])


for source_id, share in ranking(QUESTIONS["a"], plain_weights, tokenize_plain)[:6]:
    print(f"{source_id}  {share:.2f}  {'kept' if share >= 0.5 else 'dropped'} at 50%")

The garden newsletter F1 (0.54) beats the forum post A3 (0.46). At 50%, the search would read a
garden newsletter and miss the one document that contradicts the revenue figure. No threshold can
fix this: anything low enough to keep A3 also keeps F1.

The reason is in the weights printed above. In a corpus of 21 documents, "third" and "quarter" are
as rare as "Solvane": all three appear in three documents. F1 mentions the third quarter of its
gardening year and its bake sale revenue in 2025. The short forum post mentions Solvane Dynamics and
writes "Q3" instead of "third quarter". Word statistics cannot tell the *subject* of a question from
its *time period*.

### 4 What the measurement showed

Instead of guessing a better value, the threshold was measured: eight queries (the five topic
questions and three queries planned for later search rounds), and for each the range in which the
threshold separates the documents that must be found from those that must not. Squaring or cubing
the weights did not help. Two changes did:

1. **A document must name what the question is about.** If the question names a subject
   (capitalised words after the first, such as Solvane, AMSA or Brunnhof), a document must contain
   one of those names.
2. **Period abbreviations are read in full.** Financial texts write "Q3", "H1" or "FY", so the search
   expands them before comparing.

With both, every query has a clean gap, and together they leave a valid range for the threshold
**above 0.56 and at most 0.66**. The value is fixed at **0.6**. The margins are narrow, 0.04 and
0.06, which the decision record states openly and the tests guard.

In [ ]:
PERIODS = {
    "q1": "first quarter", "q2": "second quarter", "q3": "third quarter", "q4": "fourth quarter",
    "h1": "first half", "h2": "second half", "fy": "financial year",
}
RELEVANCE_CUTOFF = 0.6


def tokenize(text: str) -> set[str]:
    words: list[str] = []
    for word in re.findall(r"[a-z0-9]+", text.lower()):
        words.extend(PERIODS.get(word, word).split())
    return {singular(word) for word in words if len(word) >= 3 and word not in STOPWORDS}


def subject_words(question: str) -> set[str]:
    words = re.findall(r"[A-Za-z][A-Za-z0-9]*", question)
    names = [word for word in words[1:] if word[0].isupper() and word.lower() not in PERIODS]
    return tokenize(" ".join(names))


WEIGHTS = word_weights(CORPUS, tokenize)


def retrieve(question: str, cutoff: float = RELEVANCE_CUTOFF) -> list[SourceDoc]:
    subject = subject_words(question)
    scored = []
    for doc in CORPUS:
        words = tokenize(f"{doc['title']}\n{doc['text']}")
        if subject and not subject & words:
            continue
        score = round(sum(WEIGHTS.get(word, 0.0) for word in tokenize(question) & words), 4)
        if score > 0:
            scored.append((doc, score))
    scored.sort(key=lambda pair: (-pair[1], pair[0]["source_id"]))
    if not scored:
        return []
    best = scored[0][1]
    return [doc for doc, score in scored if score >= cutoff * best]


for topic, question in QUESTIONS.items():
    print(f"{topic}: subject {sorted(subject_words(question))} -> {[d['source_id'] for d in retrieve(question)]}")

Every question now finds exactly its own topic's documents, and the garden newsletter is gone. One
deliberate gap: topic e finds E1, E2 and E3 but not E4. E4 is a consumer study that uses different
words ("all-in charges of automated digital investment services"). Finding it takes a second,
rephrased search, which is what NB 04 builds.

The subject rule is a heuristic with known limits: a question without capitalised names gets no
subject check, and a name can be written differently in a document. A production system would use
semantic search, which compares meaning instead of words, for example with an embedding model such
as `bge-m3`. Here, word search keeps every step visible and every run identical.

### 5 Primary and secondary sources

The two collectors run the same search and split its result by document type. Splitting changes
nothing about *what* is found; it shows *where* evidence comes from, the first question a researcher
asks about a source.

In [ ]:
PRIMARY_TYPES = frozenset(
    {"press_release", "regulatory_circular", "investor_presentation_summary", "industry_report", "consumer_study"}
)


def collect_primary(state: ResearchState) -> dict:
    found = [doc for doc in retrieve(state["question"]) if doc["type"] in PRIMARY_TYPES]
    return {"documents": found, "history": [f"collect_primary: found {', '.join(d['source_id'] for d in found) or 'nothing'}"]}


def collect_secondary(state: ResearchState) -> dict:
    found = [doc for doc in retrieve(state["question"]) if doc["type"] not in PRIMARY_TYPES]
    return {"documents": found, "history": [f"collect_secondary: found {', '.join(d['source_id'] for d in found) or 'nothing'}"]}


print(collect_primary({"question": QUESTIONS["a"]})["history"])
print(collect_secondary({"question": QUESTIONS["a"]})["history"])

### 6 Reading a claim

The model reads one document at a time and answers one narrow question: which figure does this
document state in answer to the research question? It replies in a fixed JSON form, which
`read_claim` turns into the figure or `None`. The prompt log further down shows why the fixed form
is essential.

In [ ]:
CLAIM_PROMPT = ChatPromptTemplate.from_template(
    "Read the document and find the figure it states in answer to the question. "
    'Reply with JSON only, in exactly this form: {{"figure": "<the figure as written in the document>"}}. '
    'If the document does not state such a figure, reply {{"figure": null}}.\n\n'
    "Question: {question}\n\nDocument: {document}"
)


def extract_claim(question: str, doc: SourceDoc) -> str | None:
    messages = CLAIM_PROMPT.invoke({"question": question, "document": f"{doc['title']}\n\n{doc['text']}"})
    return read_claim(llm.invoke(messages).content)


DOCS_A = retrieve(QUESTIONS["a"])
claims_a = {doc["source_id"]: extract_claim(QUESTIONS["a"], doc) for doc in DOCS_A}
print(claims_a)

Two credible sources report CHF 342 million; the forum post claims CHF 310 million. The model has
only *read*. Whether the claims contradict each other, and which one to believe, is for the rules.

`parse_figure` turns a figure into a comparable value, so that "CHF 1.2 billion" and "CHF 1,200
million" or "0.62%" and "0.62% of assets" count as the same.

In [ ]:
print(parse_figure("CHF 1.2 billion"), parse_figure("0.62% of assets"), parse_figure("0.62%"))

### 7 The evidence rules

The rules, as agreed in ADR 0007:

| # | Rule | Value |
|---|---|---|
| P1 | A source counts as evidence only at or above the authority floor | 0.5 |
| P2 | The quality gate: at least two independent qualifying sources **and** summed authority of at least | 1.0 |
| P3 | Contradictions are always flagged, and resolved automatically only when the other side is weak or outdated | |
| P4 | A source is outdated when a newer source covers the same point and is at least 12 months younger | |

"Independent" means a different origin: a document that copies another (its `based_on` field names
the source) counts once together with its original.

Each rule becomes one small step. Every source that does not count gets a stated reason, because an
assessment that cannot explain itself cannot be checked.

In [ ]:
AUTHORITY_FLOOR = 0.5                     # P1
EVIDENCE_THRESHOLD = 1.0                  # P2
OUTDATED_AFTER = dt.timedelta(days=365)   # P4


def find_outdated(stated: dict, by_id: dict) -> dict[str, str]:
    dates = {sid: dt.date.fromisoformat(by_id[sid]["date"]) for sid in stated}
    outdated = {}
    for sid in stated:
        newer = [other for other in stated if dates[other] - dates[sid] >= OUTDATED_AFTER]
        if newer:
            successor = max(newer, key=lambda other: (by_id[other]["authority_weight"], dates[other]))
            outdated[sid] = f"outdated: superseded by {successor} ({dates[successor].isoformat()})"
    return outdated


def origin(doc_id: str, by_id: dict) -> str:
    seen, current = {doc_id}, doc_id
    while by_id.get(current, {}).get("based_on"):
        current = by_id[current]["based_on"][0]
        if current in seen:
            break
        seen.add(current)
    return current


def assess(documents: list[SourceDoc], claims: dict[str, str | None]) -> Assessment:
    by_id = {doc["source_id"]: doc for doc in documents}
    stated = {sid: parse_figure(fig) for sid, fig in claims.items() if fig is not None and sid in by_id}
    disregarded = {sid: "states no figure" for sid in by_id if sid not in stated}

    disregarded.update(find_outdated(stated, by_id))                          # P4
    for sid in stated:                                                         # P1
        weight = by_id[sid]["authority_weight"]
        if sid not in disregarded and weight < AUTHORITY_FLOOR:
            disregarded[sid] = f"below the authority floor ({weight:g} < {AUTHORITY_FLOOR:g})"

    contradiction = len(set(stated.values())) > 1                              # P3: always flagged
    qualifying = {sid: fig for sid, fig in stated.items() if sid not in disregarded}
    candidates = set(qualifying.values())
    if len(candidates) != 1:                                                   # none, or credible disagreement
        return Assessment(0.0, contradiction, None, [], disregarded, unresolved=len(candidates) > 1)

    (winner,) = candidates
    counted: dict[str, str] = {}
    for sid in sorted(qualifying, key=lambda s: -by_id[s]["authority_weight"]):
        root = origin(sid, by_id)                                              # independence
        if root in counted:
            disregarded[sid] = f"copy: rests on {root}, already counted via {counted[root]}"
        else:
            counted[root] = sid
    supporting = sorted(counted.values())
    score = round(sum(by_id[sid]["authority_weight"] for sid in supporting), 2)
    return Assessment(score, contradiction, format_figure(winner), supporting, disregarded)


result_a = assess(DOCS_A, claims_a)
print(result_a)
print("Contradiction exists:  ", result_a.contradiction_flag)
print("Contradiction resolved:", result_a.contradiction_resolved)
print("Quality gate passed:   ", result_a.sufficient)

Topic a: the contradiction is **flagged** (CHF 310 million against CHF 342 million) and **resolved**,
because the only source of the lower figure is an anonymous forum post below the authority floor. The
two credible sources reach 1.6, so the gate is passed.

These are three different properties, and the rest of the series keeps them apart:

- **a contradiction exists:** documents state incompatible figures (`contradiction_flag`);
- **the contradiction is resolved:** the rules still leave one supported figure, because the other
  side is weak, outdated or a copy (`contradiction_resolved`);
- **the evidence is sufficient:** the quality gate is passed (`sufficient`).

Topic a has all three. A contradiction does not stop an answer, and the absence of one does not
make the evidence enough.

### 8 Who assigns authority weights?

In this corpus, every document carries an authority weight in its front matter, assigned by hand
when the corpus was written. In practice, such judgements come from source policies of newsrooms
and research teams, from regulatory status (an authority's circular binds; a blog does not), from
ratings of data providers, and from editorial experience. They are always someone's judgement, and
the outcome depends on them: lower the floor or raise the forum post's weight, and topic a ends
differently. That is a limitation of every rule-based assessment, not only of this one. NB 05's
exercise changes the floor and watches a test case break.

### 9 Copies count once

None of the five questions shows the independence rule at work on its own, because the copied
documents in the corpus already fail another rule (E2 is below the floor, D4 is outdated). So here is
a deliberate what-if: suppose the blog E1 and the newsletter E2, which copies it, were reputable
sources with an authority of 0.7.

In [ ]:
reputable_copies = [dict(doc, authority_weight=0.7) for doc in CORPUS if doc["source_id"] in ("E1", "E2")]
what_if = assess(reputable_copies, {"E1": "0.38%", "E2": "0.38%"})
print(what_if.supporting, what_if.evidence_score, what_if.sufficient)
print(what_if.disregarded)

Two documents, one origin: the newsletter is counted as a copy of the blog, the score stays at 0.7,
and the gate fails. Counting documents instead of origins would have produced a confident 1.4.

### 10 The whole graph

The scorer reads every document's claim and applies the rules; the writer turns the assessment into
an answer that names its sources and every disregarded document with its reason.

**What changed in the contract compared with NB 02:** only `claims` is new. It is written by `score`,
alongside `evidence_score` and `contradiction_flag`. The collectors still own `documents`, and
`write_answer` still owns `draft_answer`. A new field always comes with its owner; a field without
one would be an invitation for any node to write it.

In [ ]:
def score(state: ResearchState) -> dict:
    claims = {doc["source_id"]: extract_claim(state["question"], doc) for doc in state["documents"]}
    result = assess(state["documents"], claims)
    return {
        "claims": claims,
        "evidence_score": result.evidence_score,
        "contradiction_flag": result.contradiction_flag,
        "history": [f"score: evidence score {result.evidence_score:g}, contradiction {result.contradiction_flag}, gate passed {result.sufficient}"],
    }


def write_answer(state: ResearchState) -> dict:
    result = assess(state["documents"], state["claims"])
    notes = [f"{sid} disregarded: {reason}." for sid, reason in sorted(result.disregarded.items())]
    if result.sufficient:
        head = f"{result.answer} (sources: {', '.join(result.supporting)}; evidence score {result.evidence_score:g})."
    elif result.unresolved:
        head = "No confident answer: qualifying sources contradict each other."
    elif result.answer:
        head = (f"Evidence insufficient: {result.answer} rests on {', '.join(result.supporting)} only "
                f"(evidence score {result.evidence_score:g}, threshold {EVIDENCE_THRESHOLD:.1f}).")
    else:
        head = f"Evidence insufficient: no source at or above the authority floor of {AUTHORITY_FLOOR:g} states a figure."
    return {"draft_answer": " ".join([head, *notes]), "history": ["write_answer: draft written"]}


CONTRACT = {
    "documents": {"collect_primary", "collect_secondary"},
    "claims": {"score"},
    "evidence_score": {"score"},
    "contradiction_flag": {"score"},
    "draft_answer": {"write_answer"},
    "history": {ANY_NODE},
}

builder = StateGraph(ResearchState)
builder.add_node("collect_primary", collect_primary)
builder.add_node("collect_secondary", collect_secondary)
builder.add_node("score", score)
builder.add_node("write_answer", write_answer)
builder.add_edge(START, "collect_primary")
builder.add_edge(START, "collect_secondary")
builder.add_edge(["collect_primary", "collect_secondary"], "score")
builder.add_edge("score", "write_answer")
builder.add_edge("write_answer", END)
graph = builder.compile()

results = {}
for topic, question in QUESTIONS.items():
    results[topic] = graph.invoke({"question": question})
    check_contract(graph.stream({"question": question}), CONTRACT)
    print(f"{topic}: {results[topic]['draft_answer']}\n")

Five questions, five different outcomes, each with its reasons:

| Topic | Evidence score | Gate | What happened |
|---|---|---|---|
| a | 1.6 | passed | contradiction flagged and resolved: the forum post is below the floor |
| b | 2.55 | passed | three reputable sources, one figure, no contradiction |
| c | 0.95 | **failed** | only the authority's circular states a figure; the three other documents state none |
| d | 1.65 | passed | the 2023 target is outdated; the 2025 sources decide |
| e | 0.85 | **failed** | the copied 0.38% is below the floor, and 0.62% rests on one source so far |

Topics c and e are where a linear chain stops being enough: the right next step is not "write the
answer" but "search again". NB 04 adds that loop.

## Common mistakes

### Mistake 1: retrieving every document that shares a word

Without any relevance rule, every document that shares a single word with the question is read. The
cell checks the effect: club newsletters end up in the evidence for a finance question.

In [ ]:
def retrieve_everything(question: str) -> list[str]:
    return [doc["source_id"] for doc in CORPUS if tokenize(question) & tokenize(f"{doc['title']}\n{doc['text']}")]


found = retrieve_everything("What revenue was reported for the third quarter of 2025?")
print(found)
assert {"F1", "F2"} & set(found), "expected the control topic to leak in"

The garden newsletter and the choir report land in the results: they share "revenue", "third
quarter" or "2025". Each wrong document costs a model call and, worse, could contribute a claim.
**Fix:** the subject rule and the measured threshold of experiment 4.

### Mistake 2: a document without an authority weight

The rules need every document's authority weight. A document whose front matter lacks it fails as
soon as it is turned into a state document, before any rule runs.

In [ ]:
incomplete = load_corpus()[0]
del incomplete.metadata["authority_weight"]
to_source_doc(incomplete)

*"KeyError: 'authority_weight'"*. Failing early and loudly is the right behaviour: a silent default
such as 0.5 would let an unrated document count as evidence. **Fix:** complete the front matter. The
corpus tests check every document's fields, so this cannot reach a notebook unnoticed.

### Mistake 3: reading the model's prose as data

The first version of the claim prompt (see the prompt log) let the model answer in its own words.
Code cannot read that.

In [ ]:
FIRST_CLAIM_PROMPT = ChatPromptTemplate.from_template(
    "Which figure does the document give in answer to the question?\n\n"
    "Question: {question}\n\nDocument: {document}"
)
a3 = next(doc for doc in DOCS_A if doc["source_id"] == "A3")
prose = llm.invoke(FIRST_CLAIM_PROMPT.invoke({"question": QUESTIONS["a"], "document": f"{a3['title']}\n\n{a3['text']}"})).content
read_claim(prose)

`JSONDecodeError`: the reply is a paragraph, not data. **Fix:** ask for a fixed JSON form, as the
final prompt does, and let the program fail loudly when the model ignores it rather than guess.

## Prompt log

The replies in this notebook were recorded from `qwen3:8b` (temperature 0, thinking off) on
5 October 2026 for all 18 documents the five questions retrieve.

**Version 1**

```text
Which figure does the document give in answer to the question?

Question: {question}

Document: {document}
```

**What happened.** The answers were correct but unusable as data. Each was a sentence or a
paragraph with Markdown bold, in a different form every time. Worse, the replies for the forum post
A3 and the dividend update D2 contained **two** figures each: A3's reply named the CHF 310 million
it claims and the CHF 342 million it disputes; D2's named the new 45% target and the old 60% one. A
program that picks a number out of such text cannot tell the claim from the context.

**Version 2 (final)**

```text
Read the document and find the figure it states in answer to the question.
Reply with JSON only, in exactly this form: {"figure": "<the figure as written in the document>"}.
If the document does not state such a figure, reply {"figure": null}.

Question: {question}

Document: {document}
```

**What changed.** All 18 replies were valid JSON with the right figure: 310 for A3, 45% for D2,
0.62% (not the lowest fee of 0.38%) for the market survey E3, and `null` for the three documents in
topic c that state no figure.

**Insight.** When a model's output feeds code, the format is part of the instruction. Separating
the claim from context figures is something the fixed form does and free text does not. Note what
the model was *not* asked to do: decide which figure is right. With reading alone, it made no
mistakes here; NB 05 asks it to judge.

In [ ]:
for source_id in ("A3", "D2"):
    doc = next(d for d in CORPUS if d["source_id"] == source_id)
    question = QUESTIONS["a"] if source_id == "A3" else QUESTIONS["d"]
    reply = llm.invoke(FIRST_CLAIM_PROMPT.invoke({"question": question, "document": f"{doc['title']}\n\n{doc['text']}"})).content
    print(f"--- {source_id}, first prompt ---\n{reply}\n")

With `LLM_MODE=ollama`, the next cell runs the final prompt live on every document of topic d,
including D2 with its two figures. It is skipped in scripted mode and in the tests.

In [ ]:
for doc in retrieve(QUESTIONS["d"]):
    print(doc["source_id"], extract_claim(QUESTIONS["d"], doc))

## Summary

The skeleton of NB 02 now does real work. A word search, measured rather than guessed, finds each
question's documents and keeps the club newsletters out. A model reads one figure per document in a
fixed format. Fixed rules decide what counts and say why for every source. Two of five questions do
not reach the quality gate, and that is the correct result for the evidence found so far.

Retrieval lives in the package as `retrieval.py`, the rules as `evidence.py`, and the graph as
`assistant.py`. The next cell checks that the notebook and the package agree. Retrieval and the
rules are compared in every mode, because they do not involve the model; complete graph runs are
compared only in scripted mode, because a live model can word a reply differently from one call to
the next.

In [ ]:
from research_assistant import assistant as package_assistant
from research_assistant import evidence as package_evidence
from research_assistant import retrieval as package_retrieval

package_graph = package_assistant.build_research_graph(llm)
for topic, question in QUESTIONS.items():
    package_docs = package_retrieval.retrieve(question, CORPUS, package_retrieval.word_weights(CORPUS))
    assert [d["source_id"] for d in package_docs] == [d["source_id"] for d in retrieve(question)]
    assert package_evidence.assess(results[topic]["documents"], results[topic]["claims"]) == assess(
        results[topic]["documents"], results[topic]["claims"]
    )
    if mode == "fake":
        assert package_graph.invoke({"question": question}) == results[topic]
assert package_assistant.RESEARCH_CONTRACT == CONTRACT
print("Notebook and package agree.")

**Next: NB 04, conditional routing and loops.** Topics c and e stop at the quality gate. NB 04 adds
a router that sends them back to search again, with a different query in each round, and a stop
condition that ends the search when another round cannot help.

## Exercise

1. Call `retrieve` with a question that names no subject: *"What revenue was reported for the third
   quarter of 2025?"*. Which documents come back, and why is the garden newsletter among them? What
   does this tell you about the subject rule? (Running the whole graph on this question fails in
   scripted mode: no reply was recorded for it. Why is that the right behaviour?)
2. Suppose the news report A2 had an authority weight of 0.4 instead of 0.7. Work out by hand what
   happens to topic a, then check it with `assess`. Which rule decides?